# Chapter 1: Probability and Uncertainty. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/01-prob.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/01-prob.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/01-prob.ipynb)

In [ ]:
# Packages that the exercises of this notebook use; the ones that are missing are installed
pkgs <- c("dplyr", "ggplot2", "lubridate", "quantmod")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new) > 0) install.packages(new, repos = "https://cloud.r-project.org")

# The exercises read files from data/: download the datasets of the book once
if (!dir.exists("data")) {
  options(timeout = max(3600, getOption("timeout")))
  download.file("https://github.com/VadimSokolov/bayes_ai_deep_learning/releases/latest/download/data.zip", "data.zip", mode = "wb")
  unzip("data.zip")
  unlink("data.zip")
}

## Exercise 1.42: Sampling via the Inverse CDF Method

The chapter's inverse CDF method turns uniform draws into samples from any continuous distribution: if $U \sim \text{Uniform}(0,1)$ and $F$ is a continuous, strictly increasing CDF, then $X = F^{-1}(U)$ has CDF $F$. Here you build such a sampler from scratch and check it against R's built-in generators.

a) (Pen-and-Pencil) Let $X$ have CDF $F(x) = 1 - e^{-\lambda x}$ for $x \ge 0$ (the $\text{Exponential}(\lambda)$ distribution), with $\lambda = 2$. Solve $u = F(x)$ for $x$ to obtain the inverse CDF $F^{-1}(u)$ on $u \in (0,1)$. Then, using the fact that $F$ is strictly increasing, show that if $U \sim \text{Uniform}(0,1)$ the variable $X = F^{-1}(U)$ satisfies $\mathrm{P}\left(X \le x\right) = F(x)$, so that $X$ is genuinely $\text{Exponential}(\lambda)$.

b) In R, write a function `sample_exp(n, lambda)` that draws $n$ i.i.d. $\text{Exponential}(\lambda)$ variates using only `runif()` and your formula from part (a). Do not call `rexp()` or `qexp()` inside it. Set a seed and generate $N = 10{,}000$ draws with $\lambda = 2$.

c) Compare the sample mean and variance of your draws to the theoretical $\mathrm{E}\left(X\right) = 1/\lambda$ and $\mathrm{Var}\left(X\right) = 1/\lambda^2$. Overlay a histogram of your draws on the theoretical $\text{Exponential}(\lambda)$ density (`dexp`), and produce a Q-Q plot of your sample against theoretical quantiles (`qexp` evaluated at `ppoints`).

d) Generate a second sample of size $N$ with R's built-in `rexp(N, rate = lambda)` and run a two-sample Kolmogorov-Smirnov test (`ks.test`) comparing it to your inverse-CDF sample. What do you conclude about the two generators?

e) Now let $X$ have CDF $F(x) = x^2$ on $[0,1]$, with density $f(x) = 2x$. Derive $F^{-1}(u)$ by hand, adapt your sampler, generate $N = 10{,}000$ draws, and check the sample mean and variance against the theoretical $\mathrm{E}\left(X\right) = 2/3$ and $\mathrm{Var}\left(X\right) = 1/18$.

f) (Discussion) The chapter notes that for the Normal distribution $F^{-1}$ has no closed form, so only numerical inversion is available. Generate $10{,}000$ draws via `qnorm(runif(10000))` and compare their histogram to `rnorm(10000)`. Explain why this still counts as inverse CDF sampling even though no closed-form $F^{-1}$ was derived by hand in the style of part (a).

## Exercise 1.43: The Law of Large Numbers in Action

Let $X_1, X_2, \ldots$ be i.i.d. rolls of a fair six-sided die, so $X_i \in \{1,\ldots,6\}$ each with probability $1/6$. The population mean and variance are
$$
\mu = \mathrm{E}\left(X_i\right) = 3.5, \qquad \sigma^2 = \mathrm{Var}\left(X_i\right) = \frac{35}{12} \approx 2.9167.
$$
Let $\bar X_n = \frac1n\sum_{i=1}^n X_i$ denote the running sample mean after $n$ rolls.

a) Set a seed and simulate $n=100{,}000$ die rolls. Compute the running sample mean $\bar X_n$ for every $n$ from $1$ to $100{,}000$ (the cumulative sum of the rolls divided by the running count, exactly as in the chapter's opening coin-flip figure) and plot it against $n$ on a log scale, together with a horizontal reference line at $\mu=3.5$. Repeat the simulation to generate $6$ independent sample paths and overlay them on one plot. Comment on what happens to the paths as $n$ grows.

b) For $n \in \{10, 100, 1000, 10000, 100000\}$, estimate the standard deviation of $\bar X_n$ empirically: for each $n$, simulate $M=1000$ independent replications of $\bar X_n$ and compute their sample standard deviation. Compare this empirical standard deviation to the theoretical value $\sigma/\sqrt n$, and plot both against $n$ on a log-log scale. What is the slope of the resulting line, and what does it confirm about how the spread of the sample mean shrinks with $n$?

c) (Pen-and-Pencil) Using Chebyshev's inequality, $P(|\bar X_n - \mu| > \epsilon) \le \sigma^2/(n\epsilon^2)$, find the smallest $n$ that guarantees $P(|\bar X_n - 3.5|>0.1) \le 0.05$. How does this guaranteed $n$ compare to the sample sizes in part (b) at which the simulated spread of $\bar X_n$ is already comfortably small? What does this say about how conservative the Chebyshev bound is in practice?

## Exercise 1.44: Portfolio Means, Standard Deviations and Correlation

You want to build a portfolio of exchange traded funds (ETFs) for your retirement strategy. You're thinking of whether to invest in growth or value stocks, or maybe a combination of both. Vanguard has two ETFs, one for growth (VUG) and one for value (VTV).

a) Plot the historical price series for VUG vs VTV.

b) Calculate the means and standard deviations of both ETFs.

c) Calculate their covariance.

d) Suppose you decide on a portfolio that is a 50 / 50 split. Calculate
the new mean and variance of your portfolio.

e) Which portfolio best suits you?

f) What's the probability that growth (VUG) will beat value (VTV) in
the future?

You will find the following formulas useful. Let $P$ denote the return on your portfolio which is a weighted combination $P = aX + bY$. Then
$$
E(P) = aE(X) + bE(Y )
$$
$$
Var(P ) = a^2Var(X) + b^2Var(Y ) + 2abCov(X, Y ),
$$
where $Cov(X, Y )$ is the covariance for $X$ and $Y$.

*Hint:* You can use the following code to get the data

In [ ]:
library(quantmod)
getSymbols(c("VUG","VTV"), from = "2015-01-01", to = "2024-01-01")
VUG = VUG$VUG.Adjusted
VTV = VTV$VTV.Adjusted

## Exercise 1.45: Descriptive Statistics in R

Use the `superbowl1.txt` and `derby2016.csv` datasets. The Superbowl contains data on the outcome of all previous Superbowls. The outcome is defined as the difference in scores of the favorite minus the underdog. The spread is the bookmakers' prediction of the outcome before the game begins. The Derby data consist of all of the results of the Kentucky Derby, which is run on the first Saturday in May every year at Churchill Downs racetrack. Answer the following questions

For the Superbowl data.

a) Plot the spread and outcome variables. Calculate means, standard deviations, covariances, correlations.

b) What is the mean and the standard deviation of the winning margin (outcome)?

c) Use a `boxplot` to compare the favorites' score versus the underdog.

d) Do these data look normally distributed?

For the Derby data.

a) Plot a histogram of the winning speeds and times of the horses. Why is there a long right-hand tail to the distribution of times?

b) Can you identify the outlying horse with the best winning time?

## Exercise 1.46: Berkshire Hathaway: Yahoo Finance Data

Download daily return data for Warren Buffett's firm, Berkshire Hathaway
(ticker symbol: `BRK-A`), from 1990 to the present. Analyze
these data in the following way:

a. Plot the Historical Price Performance of the stock.

b. Calculate the Daily returns. Plot a histogram of the returns. Comment on the distribution that you obtain.

c. Use the summary command to provide statistical data summaries.

d. Interpret your findings.

## Exercise 1.47: Back Cast US Presidential Elections

Use data from presidential polls to predict the winner of the elections.
We will be using data from <http://www.electoral-vote.com/>. The goal is
to use simulations to predict the winning percentage for each of the
candidates. Use `election.Rmd` script as the starter.

Report the prediction as a 50% confidence interval for each of the
candidates.

## Exercise 1.48: Russian Parliament Election Fraud

On September 28, 2016 United Russia party won a super majority of seats, which will allow them to change the Constitution without any votes of other parties. Throughout the day, there were reports of voting fraud, including video purporting to show officials stuffing ballot boxes. Additionally, results in many regions demonstrate that United Russia on many poll stations got anomalously closed results, for example, 62.2% in more than hundred poll stations in Saratov Region.

Using the assumption that United Russia's range in Saratov was [57.5%, 67.5%] and results for each poll station are rounded to one decimal point (when measure in percent), calculate probability that in 100 poll stations out of 1800 in Saratov Region the majority party got exactly 62.2%.

Do you think it can happen by chance?

## Exercise 1.49: Chicago Crime Data Analysis

On January 24, 2017, Donald Trump tweeted about \"horrible\" murder rate
in Chicago.

![Trump Tweets](https://raw.githubusercontent.com/VadimSokolov/bayes_ai_deep_learning/main/fig/trump_chi_tweet.png)

Our goal is to analyze the data and check how statistically significant such a statement. I downloaded Chicago's crime data from the data portal: [data.cityofchicago.org](https://data.cityofchicago.org). These data contain reported incidents of crime (with the exception of murders where data exist for each victim) that occurred in the City of Chicago from 2001 to present, minus the most recent seven days. Data are extracted from the Chicago Police Department's CLEAR (Citizen Law Enforcement Analysis and Reporting) system. In order to protect the privacy of crime victims, addresses are shown at the block level only, and specific locations are not identified. This data set has 6.3 million records. Each crime incident is categorized using one of the 35 primary crime types: NARCOTICS, THEFT, CRIMINAL TRESPASS, etc. I filtered incidents of type HOMICIDE into a separate data set stored in `chi_homicide.rds`. Use `chi_crime.R` as a starting script for this problem.

a) Create a heat map for the homicide incidents. In which areas of the city do you think houses are very affordable, and in which they are not?

b) Create a map by plotting a dot for each of the homicide incidents.     You will see a similar picture as you saw with the heat plot. Look at     the Hyde Park area on the south side Chicago. There is an \"island\"     with no homicide incidents! Can you explain why? Hint: You might     want to open Google maps in your browser and zoom-in into this area.

c) Though the president's tweet is consistent with the data     ([goo.gl/VTPzFw](https://goo.gl/VTPzFw)), observing 52 homicides in January     is not that unusual. Calculate the total number of homicides for     each January. Use bootstrap to estimate the 95% confidence interval for     the mean $\mu$ over January homicides. Is $52$ within the interval?     Calculate the confidence interval using $t$-ratio. Do you think results     from $t$-ratio based calculations are reliable?

d) The history of 2001-present data is rather short. Chicago tribune     provided the total number of homicides for Chicago for each month of the     1957–2014 period. Use this data set and calculate the confidence     interval for $\mu$ using bootstrap and $t$-ratio. Further answer the     following questions: (i) Assuming the monthly homicide rate follows a     Normal distribution, what is the probability that we observe 52     homicides or more? (ii) Do you think Normality assumption is     valid? (iii) Assuming the monthly homicide rate follows a Poisson     distribution, what is the probability that we observe 52 homicides     or more?

e) There is a hypothesis that crime rates are related to temperatures     ([goo.gl/nPpHwv](https://goo.gl/nPpHwv)). Check this hypothesis using simple     regression. Use a linear model to regress homicide rate to the average     maximum temperature. Does this relation appear significant? Perform     residual diagnostics and find outliers and leverage points.

f) There is another hypothesis that the rise in murder is related to the     pullback in proactive policing that started in November of 2015 as a     result of Laquan McDonald video release (<https://goo.gl/7cm1CC>,     <https://goo.gl/WcH2uB>). I calculated the total number of homicides for     each day and split the data into two parts: before and after the video     release. Using $t$-ratio, check the hypothesis $H_0$: the homicide     rate did not change after the video release.

## Exercise 1.50: AAPL vs GOOG

Download AAPL and GOOG return data from 2018 to 2024. Plot box-plot and histogram. Calculate summary statistics using `summary` function.  Describe clearly what you learn from the summary and the plots.

## Exercise 1.51: Weather Data Analysis

Use the high-resolution data from the DCA Airport weather station (`weather_dca.csv` in the data folder), which contains sixteen years of roughly hourly observations.

A useful climate metric should help understand day-to-day life impacts, going beyond simple averages. Consider questions like:

- How often will I be biking to work in below-freezing temperatures?
- How many days a year will I see the sun?
- How many days do I need a raincoat during commute times?

a) Design a climate metric that you would find personally useful.

b) Calculate this metric from the data and present it clearly.

c) Quantify and visualize the variability in your metric (year-to-year or month-to-month).

Use `lubridate` for date/time handling and `dplyr` for data manipulation.